# LinCIR trên Kaggle — Nhóm 8

Mỗi lần chạy = một **phiên** với một `MODE`. Thứ tự (phương án nhanh, xem `docs/decisions.md`):

| Phiên | MODE | Gắn thêm Input | Thời gian |
|---|---|---|---|
| U17 ✅ | `prep_eval` | — | ~1 giờ |
| U18 ✅, U18b, U19… | `train` (hàng đợi: baseline + A seed 12345/2024/7, rồi B + A+B) — tự bỏ run đã xong, tự dừng trước 12 giờ | output **mọi phiên train trước** + output U17 (Δ) | ≤ 11,3 giờ/phiên |
| U20 | `eval` | output U17 + mọi phiên train | ~2–3 giờ |

**Mỗi phiên:** Settings → GPU T4 ×2, Internet on; Add Input `btl-lincir-code` (bản mới nhất) + các output ở cột 3
(Add Input → *Your Work* → chọn notebook này → chọn đúng Version). Sửa ô *Cấu hình* → **Save Version → Save & Run All**.
Output của phiên ở tab *Output* của version; phiên sau gắn nó làm Input.

In [ ]:
# ===== Cấu hình — chỉ sửa ô này =====
MODE = "train"          # "smoke" | "prep_eval" | "train" | "eval"  ← kiểm tra trước mỗi lần Save & Run All
SEEDS = [12345, 2024, 7]
TIME_LIMIT_H = 11.3     # không bắt đầu job mới nếu ước tính xong sau mốc này (Kaggle cắt ở 12 giờ)
EST_RUN_H = 3.6         # ước tính 1 run 5.000 bước (sau khi tăng tốc nạp dữ liệu); tự cập nhật theo run đã xong
CURVE = True            # MODE="eval": đánh giá thêm checkpoint 1000..5000 của baseline/A seed 12345 (đường học)
OWNER = "Tam"

In [ ]:
# ===== Cài đặt =====
import glob, os, shutil, subprocess, sys, json, time
src = glob.glob("/kaggle/input/**/scripts/run_experiment.py", recursive=True)
assert src, "Chưa gắn dataset btl-lincir-code (Add Input)"
SRC = os.path.dirname(os.path.dirname(sorted(src)[0]))
ROOT = "/kaggle/working/BTL-LinCIR"
if not os.path.exists(ROOT):
    shutil.copytree(SRC, ROOT, ignore=shutil.ignore_patterns("runs", "feature_cache", "eval_data"))
for name in ("hf_models", "datasets"):
    tmp = f"/tmp/{name}"; os.makedirs(tmp, exist_ok=True)
    if not os.path.exists(f"{ROOT}/code/{name}"):
        os.symlink(tmp, f"{ROOT}/code/{name}")
os.makedirs(f"{ROOT}/experiments/runs", exist_ok=True)
os.chdir(ROOT); sys.path.insert(0, f"{ROOT}/scripts")
!pip install -q webdataset 2>&1 | tail -1
!python -m spacy download en_core_web_sm -q 2>&1 | tail -1
import torch, transformers
import kaggle_jobs as kj
print("python", sys.version.split()[0], "| torch", torch.__version__, "| transformers", transformers.__version__, "| GPUs", torch.cuda.device_count())
!df -h /kaggle/working /tmp | tail -2
r = subprocess.run([sys.executable, "scripts/check_encoder_equivalence.py", "--model", "openai/clip-vit-large-patch14"], capture_output=True, text=True)
print("\n".join(l for l in r.stdout.splitlines() if l.startswith("["))); assert r.returncode == 0, r.stdout[-1500:] + r.stderr[-1500:]
print("encoder check PASS")

def sh(cmd, cwd=None):
    print(">>", " ".join(map(str, cmd)), flush=True)
    p = subprocess.run(list(map(str, cmd)), cwd=cwd, capture_output=True, text=True)
    print("\n".join((p.stdout + p.stderr).splitlines()[-12:]))
    if p.returncode: raise RuntimeError(f"failed: {cmd[1] if len(cmd) > 1 else cmd}")
    return p

# ---- mang theo kết quả các phiên trước (trong Input) vào Output của phiên này, để lần sau chỉ cần gắn Version mới nhất
def carry_forward():
    keep_steps = ("phi_latest.pt", "phi_000001000.pt", "phi_000002000.pt", "phi_000003000.pt", "phi_000004000.pt")
    n_runs = 0
    for cfg in glob.glob(f"{kj.INPUT_ROOT}/**/experiments/runs/*/config.yaml", recursive=True):
        src = os.path.dirname(cfg); dst = f"experiments/runs/{os.path.basename(src)}"
        if os.path.exists(dst):
            continue
        shutil.copytree(src, dst, ignore=shutil.ignore_patterns("checkpoints"))
        for f in keep_steps:
            if os.path.exists(f"{src}/checkpoints/{f}"):
                os.makedirs(f"{dst}/checkpoints", exist_ok=True); shutil.copy(f"{src}/checkpoints/{f}", f"{dst}/checkpoints/{f}")
        n_runs += 1
    for name in ("FashionIQ", "CIRCO"):
        hit = kj.find_input(f"eval_data/{name}")
        if hit and not os.path.exists(f"/kaggle/working/eval_data/{name}"):
            shutil.copytree(hit, f"/kaggle/working/eval_data/{name}")
    for f in glob.glob(f"{kj.INPUT_ROOT}/**/experiments/feature_cache/*.pt", recursive=True):
        os.makedirs("experiments/feature_cache", exist_ok=True)
        if not os.path.exists(f"experiments/feature_cache/{os.path.basename(f)}"): shutil.copy(f, "experiments/feature_cache/")
    for f in glob.glob(f"{kj.INPUT_ROOT}/**/experiments/gap_*", recursive=True):
        if not os.path.exists(f"experiments/{os.path.basename(f)}"): shutil.copy(f, "experiments/")
    print(f"carry-forward: {n_runs} run, eval_data={os.listdir('/kaggle/working/eval_data') if os.path.exists('/kaggle/working/eval_data') else []}")
carry_forward()

def eval_data(name):  # dữ liệu đánh giá: trong phiên này, hoặc từ output phiên prep_eval
    local = f"/kaggle/working/eval_data/{name}"
    if os.path.exists(local): return local
    hit = kj.find_input(f"eval_data/{name}")
    return hit

In [ ]:
# ===== MODE smoke: chạy thử trên dữ liệu giả (đã qua ở U07) =====
if MODE == "smoke":
    sh([sys.executable, "scripts/make_toy_benchmarks.py", "--out", "experiments/toy", "--n_captions", "4096"])
    sh([sys.executable, "scripts/run_experiment.py", "configs/EXP-900-smoke.yaml", "--seed", "12345", "--owner", OWNER,
        "--suffix=-kaggle", "--cirr", "experiments/toy/CIRR", "--fiq", "experiments/toy/FashionIQ"])

In [ ]:
# ===== MODE prep_eval (U17): FashionIQ + CIRCO + COCO val, cache gallery CIRCO, Δ cho B, đánh giá EXP-000 =====
if MODE == "prep_eval":
    os.makedirs("/kaggle/working/eval_data", exist_ok=True)
    sh([sys.executable, "scripts/prepare_eval_data.py", "--what", "fiq", "coco", "circo", "--out", "/kaggle/working/eval_data",
        "--download_dir", "/tmp/dl", "--feature_cache", "experiments/feature_cache", "--clip_model_name", "large"])
    sh([sys.executable, "../scripts/modality_gap.py", "--coco_captions", "/kaggle/working/eval_data/COCO/annotations/captions_val2017.json",
        "--coco_images", "/kaggle/working/eval_data/COCO/val2017", "--limit", "5000", "--clip_model_name", "large",
        "--save_delta", "../experiments/gap_delta_large.pt", "--out", "../experiments/gap_clip_large.json"], cwd="code")
    shutil.rmtree("/kaggle/working/eval_data/COCO/val2017")      # chỉ cần cho Δ; giữ Output gọn
    shutil.rmtree("/tmp/dl", ignore_errors=True)
    sh([sys.executable, "scripts/run_experiment.py", "configs/EXP-000-official.yaml", "--seed", "0", "--owner", OWNER,
        "--fiq", eval_data("FashionIQ"), "--circo", eval_data("CIRCO")])
    print(open("experiments/registry.csv").read())

In [ ]:
# ===== MODE train: hàng đợi ưu tiên, bỏ qua run đã có trong Input; 2 run song song trên 2 T4 =====
# Gắn Input: output các phiên train trước (để bỏ qua run đã xong) + output phiên prep_eval (Δ cho B).
if MODE == "train":
    QUEUE = [(c, s) for s in SEEDS for c in ("configs/EXP-001-baseline.yaml", "configs/EXP-002-infonce.yaml")] + \
            [(c, s) for s in SEEDS for c in ("configs/EXP-003-gapnoise.yaml", "configs/EXP-004-infonce-gapnoise.yaml")]
    done = {os.path.basename(os.path.dirname(os.path.dirname(p)))   # gồm cả run đã mang theo từ Input
            for p in glob.glob("experiments/runs/*/checkpoints/phi_latest.pt")}
    todo = [(c, s) for c, s in QUEUE if f"{os.path.basename(c)[:-5]}-s{s}" not in done]
    print("đã xong (bỏ qua):", sorted(done)); print("còn lại:", [f"{os.path.basename(c)[:-5]}-s{s}" for c, s in todo])
    if any("gapnoise" in c for c, _ in todo) and not os.path.exists("experiments/gap_delta_large.pt"):
        d = kj.find_input("experiments/gap_delta_large.pt")
        assert d, "Cần gắn output phiên prep_eval (có experiments/gap_delta_large.pt)"
        shutil.copy(d, "experiments/gap_delta_large.pt")
    kj.prefetch("large")                       # tải CLIP + caption một lần trước khi chạy song song
    jobs = [(f"{os.path.basename(c)[:-5]}-s{s}",
             [sys.executable, "scripts/run_experiment.py", c, "--seed", str(s), "--owner", OWNER, "--set", "num_workers=2"])
            for c, s in todo]
    res = kj.run_parallel(jobs, gpus=(0, 1), time_limit_h=TIME_LIMIT_H, est_hours=EST_RUN_H)
    print(json.dumps(res, indent=1)); print(open("experiments/registry.csv").read())


In [ ]:
# ===== MODE eval (U20): đánh giá mọi run có trong Input trên FashionIQ + CIRCO =====
if MODE == "eval":
    FIQ, CIRCO = eval_data("FashionIQ"), eval_data("CIRCO")
    assert FIQ and CIRCO, "Cần gắn output phiên prep_eval (eval_data/)"
    cache = kj.find_input("experiments/feature_cache/circo_val_large.pt")
    os.makedirs("experiments/feature_cache", exist_ok=True)
    for f in glob.glob(os.path.join(os.path.dirname(cache), "*.pt")) if cache else []:
        shutil.copy(f, "experiments/feature_cache/")
    # chép mọi thư mục run trong Input (giữ cả EXP-000); checkpoint chỉ liên kết tới Input, không nhân đôi dung lượng
    # các run đã được carry_forward() chép vào experiments/runs (chỉ giữ checkpoint cần dùng)
    runs = sorted(os.path.abspath(os.path.dirname(os.path.dirname(p))) for p in glob.glob("experiments/runs/*/checkpoints/phi_latest.pt"))
    print(len(runs), "run:", [os.path.basename(r) for r in runs])
    def evaluated(r):  # đã có kết quả FashionIQ + CIRCO (mang theo từ phiên eval trước) -> bỏ qua
        m = f"{r}/metrics.json"
        return os.path.exists(m) and {"average_fiq_recall_at10", "circo_map_at5"} <= set(json.load(open(m)))
    print("đã đánh giá (bỏ qua):", [os.path.basename(r) for r in runs if evaluated(r)])
    jobs = []
    for r in runs:
        name = os.path.basename(r); seed = json.load(open(f"{r}/train_metrics.json"))["args"]["seed"] if os.path.exists(f"{r}/train_metrics.json") else 0
        if not evaluated(r):
            jobs.append((f"eval-{name}", [sys.executable, "scripts/run_experiment.py", f"experiments/runs/{name}/config.yaml", "--seed", str(seed),
                         "--eval_only", "--owner", OWNER, "--fiq", FIQ, "--circo", CIRCO]))
        if CURVE and name.endswith("-s12345") and ("baseline" in name or name.startswith("EXP-002")):
            for step in (1000, 2000, 3000, 4000):
                ck = f"experiments/runs/{name}/checkpoints/phi_{step:09d}.pt"
                if os.path.exists(ck) and not os.path.exists(f"experiments/runs/{name}/curve/step_{step}/metrics.json"):
                    jobs.append((f"curve-{name}-{step}", [sys.executable, "scripts/eval_per_query.py", "--phi", ck, "--clip_model_name", "large",
                                 "--fiq", FIQ, "--out_dir", f"experiments/runs/{name}/curve/step_{step}", "--feature_cache", "experiments/feature_cache"]))
    # đánh giá EXP-000 đã có từ phiên prep_eval; các job đánh giá chạy song song trên 2 GPU
    res = kj.run_parallel(jobs, gpus=(0, 1), time_limit_h=TIME_LIMIT_H, est_hours=0.3)
    print(json.dumps(res, indent=1))
    # modality gap + độ nhất quán của phi (ảnh vs văn bản) cho seed 12345 của từng biến thể, trên 5.000 cặp COCO val2017
    sh([sys.executable, "scripts/prepare_eval_data.py", "--what", "coco", "--out", "/tmp/eval_coco", "--download_dir", "/tmp/dl"])
    phis = ["--phi", "official=official-large"]
    for label, exp in (("baseline", "EXP-001-baseline"), ("A", "EXP-002-infonce"), ("B", "EXP-003-gapnoise"), ("A+B", "EXP-004-infonce-gapnoise")):
        ck = f"experiments/runs/{exp}-s12345/checkpoints/phi_latest.pt"
        if os.path.exists(ck):
            phis += ["--phi", f"{label}=../{ck}"]
    sh([sys.executable, "../scripts/modality_gap.py", "--coco_captions", "/tmp/eval_coco/COCO/annotations/captions_val2017.json",
        "--coco_images", "/tmp/eval_coco/COCO/val2017", "--limit", "5000", "--clip_model_name", "large", *phis,
        "--out", "../experiments/gap_projection_large.json"], cwd="code")
    sh([sys.executable, "scripts/rebuild_registry.py"])
    print(open("experiments/registry.csv").read())

In [ ]:
# ===== Đóng gói kết quả -> /kaggle/working/results_<MODE>_*.zip (không gồm checkpoint, dữ liệu) =====
import zipfile
stamp = time.strftime("%Y%m%d-%H%M")
out = f"/kaggle/working/results_{MODE}_{stamp}.zip"
with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as z:
    files = [p for p in glob.glob("experiments/**/*", recursive=True)
             if os.path.isfile(p) and not p.endswith(".pt") and "/checkpoints/" not in p and "/toy/" not in p and "feature_cache" not in p]
    files += glob.glob("experiments/*.pt")       # Δ (vài KB)
    for p in files:
        z.write(p, p)
print(out, f"{os.path.getsize(out) / 2**20:.1f} MB")
!du -sh /kaggle/working